# Intervention costs and return on investment data preparation

In [1]:
import json

import geopandas as gpd
import pandas as pd
import numpy as np


In [2]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"

In [3]:
locations = gpd.read_file(f"../data/processed/locations_simplified.geojson")
locations = locations[locations['type'].isin(['admin_region', 'global'])]
locations.head(2)

,id,name,type,code,geometry
0,global_sahel,Global (Sahel),global,global_sahel,"POLYGON ((42.94782 10.99465, 42.97027 10.99147..."
1,adminregion_sen,Senegal,admin_region,SEN,"POLYGON ((-17.74987 14.74329, -17.74334 14.693..."


## Costs data. 
Same data for all locations.

In [4]:
costs_df = pd.read_csv(f"{gcs_path}/wetland_costs_clean.csv")
costs_df.columns = ['wetland_type', 'restoration', 'protection', 'ratio']
costs_df.drop(columns=['ratio'], inplace=True)
# add cost for mangrove protection
costs_df.loc[costs_df['wetland_type'] == 'Mangroves', 'protection'] = 46
costs_df

,wetland_type,restoration,protection
0,Seagrass,18402.0,NaN
1,Kelp forests,27198.0,NaN
2,Coral reefs,37343.0,304.0
3,Estuaries,NaN,NaN
4,Salt marshes,28952.0,3880.0
5,Mangroves,2332.0,46.0
6,Tidal flats,5069.0,NaN
7,Lakes,NaN,NaN
8,Rivers and streams,71346.0,NaN
9,Inland marshes and swamps,24308.0,64.0


In [5]:
costs_long = pd.melt(costs_df, id_vars=['wetland_type'], var_name='intervention', value_name='cost_per_ha')
costs_long.dropna(inplace=True)
costs_long['wetland_type'] = costs_long['wetland_type'].str.lower()
costs_long['wetland_intervention'] = costs_long['wetland_type'] + "_" + costs_long['intervention']
costs_long['wetland_intervention'] = costs_long['wetland_intervention'].str.replace(" ", "_")
costs_long.reset_index(drop=True, inplace=True)
costs_long

,wetland_type,intervention,cost_per_ha,wetland_intervention
0,seagrass,restoration,18402.0,seagrass_restoration
1,kelp forests,restoration,27198.0,kelp_forests_restoration
2,coral reefs,restoration,37343.0,coral_reefs_restoration
3,salt marshes,restoration,28952.0,salt_marshes_restoration
4,mangroves,restoration,2332.0,mangroves_restoration
5,tidal flats,restoration,5069.0,tidal_flats_restoration
6,rivers and streams,restoration,71346.0,rivers_and_streams_restoration
7,inland marshes and swamps,restoration,24308.0,inland_marshes_and_swamps_restoration
8,peatlands,restoration,1094.0,peatlands_restoration
9,coral reefs,protection,304.0,coral_reefs_protection


In [11]:
costs_list = []
for i in range(len(costs_long)):
    temp_dict = {
        "id": "costs-" + str(i),
        "type": costs_long.iloc[i]['intervention'],
        "label": costs_long.iloc[i]['wetland_intervention'],
        "value": costs_long.iloc[i]['cost_per_ha'],
        "unit": "Int$/ha/yr",
        "format": "number",
        "group": "wetlands",
        "color": ""
    }
    costs_list.append(temp_dict)

In [12]:
# display as json with indentation
costs_json = json.dumps(costs_list, indent=2, default=lambda x: int(x) if isinstance(x, (np.integer, np.int64)) else x)
print(costs_json)

[
  {
    "id": "costs-0",
    "type": "restoration",
    "label": "seagrass_restoration",
    "value": 18402.0,
    "unit": "Int$/ha/yr",
    "format": "number",
    "group": "wetlands",
    "color": ""
  },
  {
    "id": "costs-1",
    "type": "restoration",
    "label": "kelp_forests_restoration",
    "value": 27198.0,
    "unit": "Int$/ha/yr",
    "format": "number",
    "group": "wetlands",
    "color": ""
  },
  {
    "id": "costs-2",
    "type": "restoration",
    "label": "coral_reefs_restoration",
    "value": 37343.0,
    "unit": "Int$/ha/yr",
    "format": "number",
    "group": "wetlands",
    "color": ""
  },
  {
    "id": "costs-3",
    "type": "restoration",
    "label": "salt_marshes_restoration",
    "value": 28952.0,
    "unit": "Int$/ha/yr",
    "format": "number",
    "group": "wetlands",
    "color": ""
  },
  {
    "id": "costs-4",
    "type": "restoration",
    "label": "mangroves_restoration",
    "value": 2332.0,
    "unit": "Int$/ha/yr",
    "format": "number"

In [13]:
pd.DataFrame(costs_long['wetland_intervention'].unique())

,0
0,seagrass_restoration
1,kelp_forests_restoration
2,coral_reefs_restoration
3,salt_marshes_restoration
4,mangroves_restoration
5,tidal_flats_restoration
6,rivers_and_streams_restoration
7,inland_marshes_and_swamps_restoration
8,peatlands_restoration
9,coral_reefs_protection


In [14]:
location_data_costs = []
for i in range(len(costs_list)):
    temp_dict = {"id": "costs-" + str(i),
                 "indicator": "cost-of-intervention",
                 "location": locations.iloc[i]['id'],
                 "data" : json.loads(json.dumps(costs_list)),
                 "locale": {"en": {"labels": {
                    'seagrass_restoration' : 'Seagrass restoration',
                    'kelp_forests_restoration': 'Kelp forests restoration',
                    'coral_reefs_restoration': 'Coral reefs restoration',
                    'salt_marshes_restoration': 'Salt marshes restoration',
                    'mangroves_restoration': 'Mangroves restoration',
                    'tidal_flats_restoration': 'Tidal flats restoration',
                    'rivers_and_streams_restoration': 'Rivers and streams restoration',
                    'inland_marshes_and_swamps_restoration': 'Inland marshes and swamps restoration',
                    'peatlands_restoration': 'Peatlands restoration',
                    'coral_reefs_protection': 'Coral reefs protection',
                    'salt_marshes_protection': 'Salt marshes protection',
                    'mangroves_protection': 'Mangroves protection',
                    'inland_marshes_and_swamps_protection': 'Inland marshes and swamps protection',
                    'peatlands_protection': 'Peatlands protection'
                 }}
                 }
                 }
    location_data_costs.append(temp_dict)



In [ ]:
location_data_costs_json = json.dumps(location_data_costs, indent=2, default=lambda x: int(x) if isinstance(x, (np.integer, np.int64)) else x)
print(location_data_costs_json)

In [15]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "costs-" and "costs_"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('costs-') and not entry['id'].startswith('costs_')]

# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in location_data_costs:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)

## Return on investment data preparation

In [16]:
mitigation_data = pd.read_csv(f"{gcs_path}/Mitigation_Country_clean.csv")
mitigation_data = mitigation_data[['ISO', 'Country',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration']]
mitigation_data.columns = mitigation_data.columns.str.replace(' ', '_').str.lower()
mitigation_data.head()

,iso,country,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
0,AFG,Afghanistan,NaN,NaN,NaN,NaN
1,ALA,Åland Islands,NaN,NaN,NaN,NaN
2,ALB,Albania,NaN,NaN,NaN,NaN
3,DZA,Algeria,NaN,NaN,NaN,NaN
4,ASM,American Samoa,NaN,NaN,NaN,NaN


In [17]:
mitigation_data_sahel = mitigation_data[mitigation_data['iso'].isin(locations['code'])].copy()
mitigation_data_sahel.reset_index(drop=True, inplace=True)

sahel_average ={'iso':'global',
                'country': 'sahel',
                'reduce_peatland_degradation': round(mitigation_data_sahel['reduce_peatland_degradation'].mean(), 4),
                'peatland_restoration': round(mitigation_data_sahel['peatland_restoration'].mean(), 4),
                'reduce_mangrove_conversion': round(mitigation_data_sahel['reduce_mangrove_conversion'].mean(), 4),
                'mangrove_restoration': round(mitigation_data_sahel['mangrove_restoration'].mean(), 4)
                }
mitigation_data_sahel = pd.concat([mitigation_data_sahel, pd.DataFrame([sahel_average])], ignore_index=True)

In [18]:
mitigation_data_sahel.tail()

,iso,country,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
22,SSD,South Sudan,NaN,NaN,NaN,NaN
23,SDN,Sudan,1740.9375,NaN,1012.1145,704.0
24,TGO,Togo,NaN,NaN,NaN,NaN
25,UGA,Uganda,1207.2067,1642.5000,NaN,NaN
26,global,sahel,1756.4318,1637.7696,1163.0082,704.0


In [19]:
mitigation_data_long = pd.melt(mitigation_data_sahel, id_vars=['iso', 'country'], var_name='intervention', value_name='mitigation')
mitigation_data_long.dropna(inplace=True)
mitigation_data_long['wetland_type'] = mitigation_data_long['intervention'].apply(
    lambda x: 'peatlands' if 'peatland' in x else ('mangroves' if 'mangrove' in x else None)
)
mitigation_data_long['intervention'] = mitigation_data_long['intervention'].apply(
    lambda x: 'restoration' if 'restoration' in x else ('protection' if 'reduce' in x else None)
)
mitigation_data_long = mitigation_data_long[['iso', 'country', 'wetland_type', 'intervention', 'mitigation']]
mitigation_data_long.tail()


,iso,country,wetland_type,intervention,mitigation
100,SEN,Senegal,mangroves,restoration,704.0
101,SLE,Sierra Leone,mangroves,restoration,704.0
102,SOM,Somalia,mangroves,restoration,704.0
104,SDN,Sudan,mangroves,restoration,704.0
107,global,sahel,mangroves,restoration,704.0


In [21]:
roi_df = mitigation_data_long.merge(costs_long, on=['wetland_type', 'intervention'], how='left')
#roi_df['intervention'] = roi_df['wetland_type'] + '_' + roi_df['intervention']
roi_df.dropna(inplace=True)
roi_df.reset_index(drop=True, inplace=True)
roi_df['roi'] = roi_df['mitigation'] / roi_df['cost_per_ha']
roi_df['roi'] = roi_df['roi'].round(4)
roi_df[roi_df['country'] == 'Cameroon']


,iso,country,wetland_type,intervention,mitigation,cost_per_ha,wetland_intervention,roi
1,CMR,Cameroon,peatlands,protection,1045.2857,610.0,peatlands_protection,1.7136
11,CMR,Cameroon,peatlands,restoration,1978.6957,1094.0,peatlands_restoration,1.8087
21,CMR,Cameroon,mangroves,protection,1470.9252,46.0,mangroves_protection,31.9766
38,CMR,Cameroon,mangroves,restoration,704.0000,2332.0,mangroves_restoration,0.3019


In [22]:
roi_df['country'].unique()

array(['Benin', 'Cameroon', 'Central African Republic', 'Ethiopia',
       'Kenya', 'Nigeria', 'Sudan', 'Uganda', 'sahel', 'Burkina Faso',
       'Mali', 'Mauritania', "Côte d'Ivoire", 'Djibouti', 'Eritrea',
       'Gambia, The', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Liberia',
       'Senegal', 'Sierra Leone', 'Somalia'], dtype=object)

In [23]:
roi_list = []
for country in roi_df['country'].unique():
    roi_country = roi_df[roi_df['country'] == country]
    roi_country.reset_index(drop=True, inplace=True)
    for i in range(len(roi_country)):

        if country == 'sahel':
            location_id = "global_sahel"
        else:
            location_id = "adminregion_" + roi_country.iloc[i]['iso'].lower()
        data_list = []
        for j in range(len(roi_country)):
            data_list.append({
                "id": "roi-" + roi_country.iloc[j]['iso'].lower() + "-" + str(j),
                "type": roi_country.iloc[j]['intervention'],
                "label": roi_country.iloc[j]['wetland_intervention'],
                "value": roi_country.iloc[j]['roi'],
                "unit": "Mitigation per dollar (tCO2e/Int$/ha/yr)",
                "format": "number",
                "group": "wetlands",
                "color": ""
            })

        data_json = json.dumps(data_list, indent=2)

    temp_dict = {"id": "roi-" + roi_country.iloc[0]['iso'].lower(),
                 "indicator": "return-on-investment",
                 "location": location_id,
                 "data" : json.loads(data_json),
                 "locale": {"en": {"labels": {
                 "mangroves_protection": "Mangroves Protection",
                 "mangroves_restoration": "Mangroves Restoration",
                 "peatlands_protection": "Peatlands Protection",
                 "peatlands_restoration": "Peatlands Restoration"

                 }}}}
    roi_list.append(temp_dict)

In [24]:
print(json.dumps(roi_list, indent=2))

[
  {
    "id": "roi-ben",
    "indicator": "return-on-investment",
    "location": "adminregion_ben",
    "data": [
      {
        "id": "roi-ben-0",
        "type": "protection",
        "label": "peatlands_protection",
        "value": 4.2049,
        "unit": "Mitigation per dollar (tCO2e/Int$/ha/yr)",
        "format": "number",
        "group": "wetlands",
        "color": ""
      },
      {
        "id": "roi-ben-1",
        "type": "restoration",
        "label": "peatlands_restoration",
        "value": 2.9068,
        "unit": "Mitigation per dollar (tCO2e/Int$/ha/yr)",
        "format": "number",
        "group": "wetlands",
        "color": ""
      },
      {
        "id": "roi-ben-2",
        "type": "protection",
        "label": "mangroves_protection",
        "value": 18.4135,
        "unit": "Mitigation per dollar (tCO2e/Int$/ha/yr)",
        "format": "number",
        "group": "wetlands",
        "color": ""
      },
      {
        "id": "roi-ben-3",
        "type"

In [25]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "roi-"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('roi-')]
# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in roi_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2, default=lambda x: int(x) if isinstance(x, (np.integer, np.int64)) else x)